# 🏭 Celeris Automobili Industrial: Demand Forecasting & Planning Control Tower
### MKS AIML Hackathon Phase 2: Super-Expert End-to-End Demand Intelligence Platform

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pkprajapati7402/mks-hackathon/blob/main/mks_datathon_notebook.ipynb)
[![Python 3.10+](https://img.shields.io/badge/Python-3.10%2B-3776AB?logo=python&logoColor=white)](https://www.python.org/)
[![LightGBM](https://img.shields.io/badge/LightGBM-4.3%2B-brightgreen)](https://lightgbm.readthedocs.io/)
[![19-Month Forecast](https://img.shields.io/badge/Forecast%20Horizon-Nov%202026%20--%20May%202028-orange)](#)
[![Submission Integrity](https://img.shields.io/badge/Submission%20Rows-58%2C219%20Valid-success)](#)

---
### 📌 Platform Architecture & Differentiators
Built for **Celeris Automobili Industrial** across **601 SKUs, 7 Product Families, 4 Sales Channels, and 6 Global Regions** (`2019-01 → 2028-05`):
1. **Exact Arithmetic & Currency Triangulation (`Units × Price = Revenue`)**: Recovers 10×/100× unit typos, cents entries, and EUR→USD currency mismatches while preserving legitimate H2-2022 6% surcharges (`EV-027`).
2. **Master Data Contradictions & Rule-6 Cadence Shift Audits**: Documents family/BU mismatches, Rule-5 launch/EOL violations, lost accounts, SKU switches, and temporal customer cadence shifts.
3. **Scoped Business Events Engine (`EV-001`..`EV-068`)**: Models open-ended `EV-052` (Helios India line expansion), `EV-062` (Orion NA July–Aug 2027 planned retooling), `EV-033` (LatAm ERP gap), `EV-039` (Red Sea), Ramadan, LNY, and price-hike announcements.
4. **Zero-Leakage Dual-Objective Ensemble + Conformal Quantiles (CQR)**: Blends LightGBM Tweedie + Log1p-L1 + Seasonal-EWM Anchor trained over causal multi-origin snapshots, with horizon-stratified CQR achieving ~80% `[P10, P90]` empirical coverage.
5. **Native Tree-SHAP Explainability & ABC-XYZ Working Capital Optimizer**: Decomposes every SKU-Region forecast into 6 business drivers (`pred_contrib=True`) and optimizes safety stock capital across the 9-cell ABC×XYZ matrix.

## 1. Environment Verification & Auto-Setup (Google Colab Cloud & Local)
Detects Colab Web, VS Code Colab Cloud, or Local runtime; syncs `/content/mks-hackathon`; installs dependencies; and configures Plotly & `sys.path`.

In [ ]:
# -----------------------------------------------------------------------------
# 1. COLAB CLOUD & LOCAL ENVIRONMENT AUTO-SETUP
# -----------------------------------------------------------------------------
import os
import sys
import shutil
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/pkprajapati7402/mks-hackathon.git"
COLAB_REPO = Path("/content/mks-hackathon")

def detect_colab() -> bool:
    if "google.colab" in sys.modules or any(k.startswith("COLAB_") for k in os.environ):
        return True
    if Path("/content").exists():
        return True
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

IN_COLAB = detect_colab()

if IN_COLAB:
    print("⚡ Google Colab Cloud runtime detected.")
    if not (Path.cwd() / "src").exists():
        if (COLAB_REPO / ".git").exists() and (COLAB_REPO / "src").exists():
            print(f"🔄 Syncing existing clone at {COLAB_REPO} to latest origin/main...")
            subprocess.run(["git", "-C", str(COLAB_REPO), "fetch", "-q", "origin", "main"], check=False)
            subprocess.run(["git", "-C", str(COLAB_REPO), "reset", "-q", "--hard", "origin/main"], check=False)
        else:
            print(f"📥 Cloning repository from {REPO_URL}...")
            if COLAB_REPO.exists():
                shutil.rmtree(str(COLAB_REPO), ignore_errors=True)
            subprocess.run(["git", "clone", "-q", REPO_URL, str(COLAB_REPO)], check=True)
        %cd /content/mks-hackathon
        os.chdir("/content/mks-hackathon")
    else:
        subprocess.run(["git", "fetch", "-q", "origin", "main"], check=False)
        subprocess.run(["git", "reset", "-q", "--hard", "origin/main"], check=False)

    print("📦 Verifying/installing production dependencies...")
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q",
        "pyarrow>=15.0.0", "lightgbm>=4.3.0", "streamlit>=1.35.0",
        "plotly>=5.20.0", "statsmodels>=0.14.0", "scikit-learn>=1.4.0", "tabulate>=0.9.0"
    ], check=True)
else:
    print(f"💻 Local environment detected: {Path.cwd()}")

PROJECT_ROOT = Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import plotly.io as pio
pio.renderers.default = "colab" if IN_COLAB else "notebook_connected"

try:
    from IPython.display import display
except ImportError:
    display = print

import json
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

assert (PROJECT_ROOT / "run_pipeline.py").exists(), f"run_pipeline.py missing in {PROJECT_ROOT}"
assert (PROJECT_ROOT / "student_data" / "train.csv").exists(), f"student_data/train.csv missing in {PROJECT_ROOT}"

print(f"\n✅ Environment Ready!")
print(f"   - Working Dir:     {PROJECT_ROOT}")
print(f"   - Python Version:  {sys.version.split()[0]}")
print(f"   - Plotly Renderer: {pio.renderers.default}")

## 2. Execute Master Demand Pipeline
Runs all 6 stages end-to-end: Triangulated Cleaning → Anomaly & Cycle Intelligence → Causal Multi-Horizon Features → Leakage-Free Rolling Backtesting → Hybrid LightGBM + CQR + Tree-SHAP → Hierarchical Reconciliation + ABC-XYZ Matrix.

In [ ]:
# -----------------------------------------------------------------------------
# 2. EXECUTE MASTER DEMAND FORECASTING PIPELINE
# -----------------------------------------------------------------------------
import os
import sys
import shutil
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/pkprajapati7402/mks-hackathon.git"
COLAB_REPO = Path("/content/mks-hackathon")

if not (Path.cwd() / "run_pipeline.py").exists():
    if (COLAB_REPO / "run_pipeline.py").exists():
        %cd /content/mks-hackathon
        os.chdir("/content/mks-hackathon")
    elif Path("/content").exists() or "google.colab" in sys.modules or any(k.startswith("COLAB_") for k in os.environ):
        print("⚡ Auto-cloning & configuring repository on Google Colab Cloud...")
        if COLAB_REPO.exists():
            shutil.rmtree(str(COLAB_REPO), ignore_errors=True)
        subprocess.run(["git", "clone", "-q", REPO_URL, str(COLAB_REPO)], check=True)
        %cd /content/mks-hackathon
        os.chdir("/content/mks-hackathon")
        subprocess.run([
            sys.executable, "-m", "pip", "install", "-q",
            "pyarrow>=15.0.0", "lightgbm>=4.3.0", "scipy>=1.13.0", "streamlit>=1.35.0",
            "plotly>=5.20.0", "statsmodels>=0.14.0", "scikit-learn>=1.4.0", "tabulate>=0.9.0"
        ], check=True)

PROJECT_ROOT = Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from run_pipeline import run_pipeline

run_pipeline(stage="all")

## 3. Data Integrity, Master Contradictions & 10-Category Anomaly Register

In [ ]:
import os
import sys
import json
import pandas as pd
from pathlib import Path
try:
    from IPython.display import display
except ImportError:
    display = print
if not (Path.cwd() / 'src').exists() and Path('/content/mks-hackathon/src').exists():
    os.chdir('/content/mks-hackathon')
if str(Path.cwd().resolve()) not in sys.path:
    sys.path.insert(0, str(Path.cwd().resolve()))
from src.config import REPORTS_DIR

with open(REPORTS_DIR / "data_quality_report.json", "r", encoding="utf-8") as f:
    dq_report = json.load(f)

train_dq = dq_report.get("file_summaries", {}).get("train.csv", {})
print(f"Exact Arithmetic & Currency Triangulation (train.csv):")
print(f"  - 10x Unit Typos Recovered: {train_dq.get('unit_10x_recovered', 0):,} | 100x Unit Typos: {train_dq.get('unit_100x_recovered', 0):,}")
print(f"  - Cents (100x) Prices: {train_dq.get('cents_100x_fixed', 0):,} | Decimal (10x): {train_dq.get('decimal_10x_fixed', 0):,} | EUR->USD: {train_dq.get('currency_eur_fixed', 0):,}")

file_summaries = [
    {"Dataset": fname, "Raw Rows": f"{stats.get('raw_rows', 0):,}", "Cleaned Rows": f"{stats.get('clean_rows', 0):,}", "Status": "Verified Clean ✅"}
    for fname, stats in dq_report.get("file_summaries", {}).items()
]
display(pd.DataFrame(file_summaries))

contra_df = pd.read_csv(REPORTS_DIR / "master_data_contradictions.csv")
print(f"\nMaster Data vs Transactions Contradictions Resolved: {len(contra_df):,} SKU-Rule groups")
display(contra_df.head(8))

anomaly_df = pd.read_csv(REPORTS_DIR / "anomaly_register.csv")
print(f"\nTotal Anomalies Categorized: {len(anomaly_df):,}")
display(anomaly_df["Anomaly_Type"].value_counts().to_frame("Count"))
cols = [c for c in ["Date", "SKU_ID", "Region", "Customer_Name", "Anomaly_Type", "Units_Sold", "Units_Sold_Unconstrained", "Lost_Units", "Reason", "Action_Taken"] if c in anomaly_df.columns]
display(anomaly_df[cols].head(8))

## 4. Econometric Drivers, Rule-6 Ordering Cadence Shifts & EV Shift

In [ ]:
import os
import sys
import pandas as pd
import plotly.express as px
from pathlib import Path
try:
    from IPython.display import display
except ImportError:
    display = print
if not (Path.cwd() / 'src').exists() and Path('/content/mks-hackathon/src').exists():
    os.chdir('/content/mks-hackathon')
if str(Path.cwd().resolve()) not in sys.path:
    sys.path.insert(0, str(Path.cwd().resolve()))
from src.config import REPORTS_DIR

macro_lag_df = pd.read_csv(REPORTS_DIR / "macro_lead_lag_analysis.csv")
print("Top Macro Lead/Lag Relationships:")
display(macro_lag_df.reindex(macro_lag_df["Max_Correlation"].abs().sort_values(ascending=False).index)[[
    "Product_Family", "Macro_Indicator", "Optimal_Lag_Months", "Max_Correlation", "Relationship"
]].head(10))

cad_changes = pd.read_csv(REPORTS_DIR / "cadence_changes.csv")
print(f"\nCustomer Ordering-Cycle Shifts Detected (Rule 6, 2019–22 vs 2023–26): {len(cad_changes):,}")
display(cad_changes["Shift_Pattern"].value_counts().head(8).to_frame("Customer-SKU Count"))

ev_df = pd.read_csv(REPORTS_DIR / "ev_shift_analysis.csv")
display(ev_df)
fig_ev = px.bar(
    ev_df, x="Year", y=["ICE_Volume", "EV_Volume"], barmode="group",
    title="Powertrain Paradigm Shift: Annual Unconstrained Volume (ICE vs EV)",
    color_discrete_map={"ICE_Volume": "#EF4444", "EV_Volume": "#10B981"}, template="plotly_dark"
)
fig_ev.update_layout(paper_bgcolor="#1E293B", plot_bgcolor="#0F172A")
fig_ev.show()

## 5. Zero-Leakage Rolling-Origin Backtest Tournament & CQR Coverage

In [ ]:
import os
import sys
import pandas as pd
import plotly.express as px
from pathlib import Path
try:
    from IPython.display import display
except ImportError:
    display = print
if not (Path.cwd() / 'src').exists() and Path('/content/mks-hackathon/src').exists():
    os.chdir('/content/mks-hackathon')
if str(Path.cwd().resolve()) not in sys.path:
    sys.path.insert(0, str(Path.cwd().resolve()))
from src.config import OUTPUTS_DIR

metrics_df = pd.read_csv(OUTPUTS_DIR / "backtest_metrics.csv")
print("Zero-Leakage Rolling-Origin 19-Month Backtest Performance:")
display(metrics_df)

fig_bt = px.bar(
    metrics_df, x="Origin", y="WAPE_Pct", color="Model", barmode="group",
    title="19-Month Out-of-Sample Backtest WAPE (%) by Origin & Model",
    template="plotly_dark",
    color_discrete_map={
        "Grandmaster Ensemble P50": "#00E5FF", "LightGBM Tweedie": "#38BDF8", "LightGBM LogRatio": "#2DD4BF",
        "Hurdle Two-Stage": "#10B981", "HistGB Poisson": "#818CF8", "PyTorch Entity-ResNet": "#EC4899",
        "Seasonal Naive": "#F59E0B", "Moving Average": "#64748B", "Intermittent SBA": "#A855F7"
    }
)
fig_bt.update_layout(paper_bgcolor="#1E293B", plot_bgcolor="#0F172A", yaxis_title="WAPE (%)")
fig_bt.show()

bd_df = pd.read_csv(OUTPUTS_DIR / "backtest_breakdowns.csv")
hor_df = bd_df[(bd_df["Origin"] == "2025-03") & (bd_df["Breakdown_Type"] == "Horizon")].copy()
if not hor_df.empty:
    hor_df["Horizon_Month"] = hor_df["Group"].astype(int)
    hor_df = hor_df.sort_values("Horizon_Month")
    fig_h = px.line(
        hor_df, x="Horizon_Month", y=["LightGBM_WAPE", "Seasonal_Naive_WAPE", "Coverage_Pct"],
        title="Holdout 2025-03 Accuracy & Interval Coverage Across Month 1 → Month 19", template="plotly_dark"
    )
    fig_h.update_layout(paper_bgcolor="#1E293B", plot_bgcolor="#0F172A")
    fig_h.show()

selection_df = pd.read_csv(OUTPUTS_DIR / "family_model_selection.csv")
display(selection_df)

## 6. Continuous Horizon (`2019-2028`), Native Tree-SHAP Attribution & ABC-XYZ Matrix

In [ ]:
import os
import sys
import pandas as pd
import plotly.express as px
from pathlib import Path
try:
    from IPython.display import display
except ImportError:
    display = print
if not (Path.cwd() / 'src').exists() and Path('/content/mks-hackathon/src').exists():
    os.chdir('/content/mks-hackathon')
if str(Path.cwd().resolve()) not in sys.path:
    sys.path.insert(0, str(Path.cwd().resolve()))
from src.config import CLEANED_DATA_DIR, OUTPUTS_DIR
from src.dashboard.components import plot_continuous_timeline

train_df = pd.read_parquet(CLEANED_DATA_DIR / "unconstrained_demand.parquet")
fcst_df = pd.read_parquet(OUTPUTS_DIR / "forecast_intervals.parquet")

hist_monthly = train_df.groupby("Date").agg({"Units_Sold": "sum", "Units_Sold_Unconstrained": "sum"}).reset_index()
fcst_monthly = fcst_df.groupby("Date").agg({"P10": "sum", "P50": "sum", "P90": "sum", "Units_Sold_Forecast": "sum"}).reset_index()

fig_timeline = plot_continuous_timeline(
    hist_monthly, fcst_monthly,
    title="Celeris Automobili Continuous Demand Horizon (2019–2028) with Calibrated P10/P90 Ribbon"
)
fig_timeline.show()

# Verify Month 1..19 continuity (zero Month-13 collapse)
print("19-Month Forecast Continuity Check (Units/Month):")
display(fcst_monthly[["Date", "P10", "Units_Sold_Forecast", "P90"]])

# Tree-SHAP Business Driver Attribution
drv_path = OUTPUTS_DIR / "forecast_drivers.parquet"
if drv_path.exists():
    drv_df = pd.read_parquet(drv_path)
    d_cols = ["Baseline & Seasonal Level", "Macro Economy", "EV Transition", "Calendar & Price Hikes", "Customer & Scoped Events", "Catalog & Lifecycle"]
    drv_abs = pd.DataFrame([{"Driver": c, "Absolute_Units": drv_df[c].abs().sum(), "Net_Signed": drv_df[c].sum()} for c in d_cols if c in drv_df.columns])
    fig_shap = px.bar(drv_abs.sort_values("Absolute_Units"), x="Absolute_Units", y="Driver", orientation="h", color="Net_Signed", color_continuous_scale="RdYlGn", title="Native LightGBM Tree-SHAP Driver Attribution Across 58,219 Forecasts", template="plotly_dark")
    fig_shap.update_layout(paper_bgcolor="#1E293B", plot_bgcolor="#0F172A")
    fig_shap.show()

# ABC-XYZ Portfolio Working Capital Matrix
abc_path = OUTPUTS_DIR / "abc_xyz_inventory.csv"
if abc_path.exists():
    abc_df = pd.read_csv(abc_path)
    abc_mat = abc_df.groupby(["ABC_Class", "XYZ_Class"])["SKU_ID"].count().unstack(fill_value=0).reindex(index=["A", "B", "C"], columns=["X", "Y", "Z"], fill_value=0)
    fig_abc = px.imshow(abc_mat, text_auto=True, color_continuous_scale="Teal", title="ABC (Revenue Tier) × XYZ (Demand Variability) Portfolio Matrix (SKU Counts)", template="plotly_dark")
    fig_abc.update_layout(paper_bgcolor="#1E293B")
    fig_abc.show()

## 7. Official Competition Submission Verification (58,219 Rows)

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display = print
if not (Path.cwd() / 'src').exists() and Path('/content/mks-hackathon/src').exists():
    os.chdir('/content/mks-hackathon')
if str(Path.cwd().resolve()) not in sys.path:
    sys.path.insert(0, str(Path.cwd().resolve()))
from src.config import RAW_TEST, OUTPUTS_DIR

sub_path = OUTPUTS_DIR / "submission.csv"
if not sub_path.exists():
    sub_path = Path("submission.csv")

sub_df = pd.read_csv(sub_path)
raw_test = pd.read_csv(RAW_TEST, low_memory=False)
recon_df = pd.read_csv(OUTPUTS_DIR / "hierarchical_reconciliation.csv")

print("=" * 60)
print("  OFFICIAL SUBMISSION INTEGRITY CHECKLIST")
print("=" * 60)

assert len(sub_df) == 58219, f"Row count mismatch! Expected 58,219, got {len(sub_df)}"
print(f"✅ Row Count: {len(sub_df):,} matches test.csv exactly.")

expected_cols = ["Date", "SKU_ID", "Region", "Units_Sold_Forecast"]
assert list(sub_df.columns) == expected_cols, f"Column mismatch: {list(sub_df.columns)}"
print(f"✅ Columns: {list(sub_df.columns)} verified.")

assert sub_df.isnull().sum().sum() == 0, "Null values found!"
print("✅ Null Count: 0 missing or NaN values.")

assert (sub_df["Units_Sold_Forecast"] >= 0).all(), "Negative forecasts found!"
assert (sub_df["Units_Sold_Forecast"] == sub_df["Units_Sold_Forecast"].round()).all(), "Non-integer forecasts found!"
print("✅ Validity: 100% non-negative integers.")

alignment_diff = (
    (sub_df["Date"] != raw_test["Date"]).sum()
    + (sub_df["SKU_ID"] != raw_test["SKU_ID"]).sum()
    + (sub_df["Region"] != raw_test["Region"]).sum()
)
assert alignment_diff == 0, f"Key alignment mismatch in {alignment_diff} rows!"
print("✅ Alignment: 100% key-aligned with student_data/test.csv.")

assert (recon_df["Discrepancy"] == 0).all(), "Reconciliation discrepancy != 0!"
print("✅ Hierarchy Reconciliation: 0 discrepancy across BU, Family, and Region sums.")

print(f"\n📊 Total 19-Month Forecast Demand: {sub_df['Units_Sold_Forecast'].sum():,} units")
print(f"   Monthly Average: {sub_df.groupby('Date')['Units_Sold_Forecast'].sum().mean():,.0f} units/month")
display(sub_df.head(10))

if "google.colab" in sys.modules:
    try:
        from google.colab import files
        print("\n⬇️ Triggering download of submission.csv...")
        files.download(str(sub_path))
    except Exception as exc:
        print(f"\nℹ️ Saved at {sub_path.resolve()} (direct browser download unavailable in this client: {exc})")

## 8. Launch Interactive 14-View Streamlit Control Tower
Exposes `app.py` cleanly over **Cloudflare Quick Tunnel (`trycloudflare.com`)** so all Streamlit JS modules, Plotly charts, Tree-SHAP waterfalls, and ABC-XYZ heatmaps render cleanly with **no password wall**.

In [ ]:
import os
import re
import sys
import time
import subprocess
import urllib.request
from pathlib import Path

is_colab_cloud = "google.colab" in sys.modules or any(k.startswith("COLAB_") for k in os.environ) or Path("/content").exists()

if is_colab_cloud:
    if not (Path.cwd() / "app.py").exists() and Path("/content/mks-hackathon/app.py").exists():
        %cd /content/mks-hackathon
        os.chdir("/content/mks-hackathon")

    subprocess.run(["pkill", "-f", "streamlit run"], check=False)
    subprocess.run(["pkill", "-f", "cloudflared"], check=False)
    time.sleep(1)

    st_log = open("streamlit.log", "w", encoding="utf-8")
    st_proc = subprocess.Popen(
        [
            sys.executable, "-m", "streamlit", "run", "app.py",
            "--server.port", "8501",
            "--server.address", "0.0.0.0",
            "--server.headless", "true",
            "--server.enableCORS", "false",
            "--server.enableXsrfProtection", "false",
            "--server.enableWebsocketCompression", "false",
        ],
        stdout=st_log,
        stderr=subprocess.STDOUT,
    )

    ready = False
    for _ in range(25):
        try:
            resp = urllib.request.urlopen("http://127.0.0.1:8501/_stcore/health", timeout=2).read().decode()
            if "ok" in resp.lower():
                ready = True
                break
        except Exception:
            time.sleep(1)

    if not ready:
        print("❌ Streamlit failed to start. Log output:")
        st_log.flush()
        print(Path("streamlit.log").read_text(encoding="utf-8", errors="ignore"))
    else:
        cf_bin = Path("/tmp/cloudflared")
        if not cf_bin.exists():
            print("📥 Downloading cloudflared tunnel client...")
            subprocess.run([
                "wget", "-q",
                "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
                "-O", str(cf_bin)
            ], check=True)
            os.chmod(str(cf_bin), 0o755)

        cf_proc = subprocess.Popen(
            [str(cf_bin), "tunnel", "--url", "http://127.0.0.1:8501", "--no-autoupdate"],
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
        )

        tunnel_url = None
        deadline = time.time() + 25
        while time.time() < deadline:
            line = cf_proc.stdout.readline()
            if not line:
                break
            m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
            if m:
                tunnel_url = m.group(0)
                break

        if tunnel_url:
            print("=" * 72)
            print("🚀 CELERIS CONTROL TOWER DASHBOARD READY!")
            print(f"🌐 Open Dashboard URL (No Password Required): {tunnel_url}")
            print("=" * 72)
            print("Keep this cell running while using the dashboard; click Stop to shut down.")
            try:
                cf_proc.wait()
            except KeyboardInterrupt:
                print("\n🛑 Shutting down tunnel and Streamlit server...")
                cf_proc.terminate()
                st_proc.terminate()
        else:
            print("⚠️ Could not retrieve trycloudflare URL; falling back to localtunnel...")
            cf_proc.terminate()
            !npx -y localtunnel --port 8501
else:
    print("💻 To run the 14-view Control Tower locally, execute in terminal:")
    print("   python -m streamlit run app.py")